&nbsp;

# PRO-seq related Metadata validated and modified process

## Background

- lncRNA が転写されているのか、bulk RNA-seq だけではない観点から検証するため、GENCODE v49 をアノテーションとして **PRO-seqのメタ解析** を実行しようと考えている
- これまで、いくつか熱ストレスに関連しているPRO-seqのデータや、意識してこなかったが、その他にもメタ解析ができるくらいのPRO-seqのデータが蓄積している可能性がある。
- そのため、ここでは、これまで取得されてきたPRO−seqのメタデータの処理を移行し、まとめる他、新たに検索し、使用できそうなデータもここに記載する
- (2026-02-21) GRO-seqのデータもこちらに記載する
- (2026-07-11) `bigwigaverageoverbed` ツールで、`fanta.bio`のCRE アノテーションと PRO-seq のピークを統合する
- (2026-07-22) adapter を調査する (e.g., [Illumina TruSeq-SmallRNA adapter](https://support-docs.illumina.com/SHARE/AdapterSequences/Content/TruSeq-SmallRNA.htm))

In [1]:
import sys
import polars as pl
import datetime
import pathlib
from typing import Any
sys.path.append(str(pathlib.Path('../../').resolve()))
from scripts.metadata_pairing import create_paired_metadata
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_rows(4)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


In [3]:
# drop columns list for iseq metadata
drop_columns_list = [
    "sra_aspera",
    "fastq_galaxy",
    "sra_galaxy",
    "fastq_ftp",
    "sra_ftp",
    "fastq_aspera",
    "tax_lineage",
    "bam_md5",
    "fastq_md5",
    "sra_md5"
]

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. GSE228346 PRO-seq (SINGLE)

- ✅ DOWNLOAD
- ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD)
- ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD)

&nbsp;

- validate date: 2025-05-02
- validate date: 2026-02-19

&nbsp;

- [12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb) で得られたデータ
- [TFIIH kinase CDK7 drives cell proliferation through a common core transcription factor network](https://doi.org/10.1126/sciadv.adr9660)
- DMSOで処理されたコントロールと42℃で処理された熱ストレスのサンプル (例: `DMSO_0_R1, DMSO_20_R1`) と、CDK阻害剤である `SY5609` を処理された状態でのコントロールと熱ストレスの状態 ( `SY5609_0_R1, SY5609_20_R1`) が存在する

&nbsp;

### Adapter

- 以下のような記述がある
- `A single-ended library preparation using Illumina TruSeq Small RNA adapters was prepared as described in Mahat et al. (62) and adapter clean-up was performed with 1× AMPure XP beads (Beckman A63880).`

In [4]:
GSE228346_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE228346/GSE228346_PEP/GSE228346_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("KAPA mRNA HyperPrep Kit").alias("rna_library"), # add rna_library column
    pl.lit("https://doi.org/10.1126/sciadv.adr9660").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# PRO-seq metadata
GSE228346_PEP_pro_seq = GSE228346_PEP.filter(
    (pl.col("sample_library_selection") == "other") &
    (pl.col("library_strategy") == "PRO-seq")
).with_columns(
    pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")
)

# metadata from iseq
GSE228346_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE228346_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE228346_iseq_filtered = GSE228346_iseq[[s.name for s in GSE228346_iseq if not (s.null_count() == GSE228346_iseq.height)]]

In [5]:
GSE228346_pro_seq_combined = GSE228346_PEP_pro_seq.join(
    GSE228346_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE228346_pro_seq_combined.write_csv(
    "../../Data/metadata/metadata_proseq/GSE228346_pro_seq_modified_metadata.tsv",
    separator='\t'
)

pl.Config.set_tbl_rows(2)
display(GSE228346_pro_seq_combined)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,time,molecule_subtype,library_strategy,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,adapter,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy_right,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,library_name,first_created,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""dmso_0_r1_pro-seq_""",null,null,null,null,null,null,"""DMSO_0_R1 [PRO-seq]""","""GSM7119231""","""Public on Dec 19 2024""","""Mar 28 2023""","""Dec 19 2024""","""SRA""",1,"""wild type OV90 cells""","""Homo sapiens""",9606,"""total RNA""","""PRO-seq from OV90 cells, proseq_virtual_spikein_normalized_counts.txt""","""GPL24676""","""Taylor,K,Jones""","""tajo5912@colorado.edu""",3606246612,"""University of Colorado Boulder""","""3415 Colorado Ave""","""Boulder""","""CO""",80301,"""USA""","""Illumina NovaSeq 6000""","""other""","""transcriptomic""","""OTHER""","""GSM7119231""","""NONE""","""GSE228346""",0,"""SRX19793272""","""Wild-type OV90 cells""","""ovarian cancer""","""30min pretreatment with DMSO at 37C followed by a 42C heat shock for 0min with a 0min recovery at 37C""",30,"""Nascent RNA""","""PRO-seq""","""hg38""","""NOT PROVIDED""","""SAMN33946087""","""KAPA mRNA HyperPrep Kit""","""https://doi.org/10.1126/sciadv.adr9660""","""Illumina TruSeq Small RNA adapters""","""SRR23989399""","""Illumina NovaSeq 6000 sequencing: GSM7119231: DMSO_0_R1 [PRO-seq] Homo sapiens OTHER""","""The TFIIH kinase CDK7 governs RNA polymerase II initiation, elongation, termination, and RNA processing""","""SRA1611893""","""other""","""2024-12-29""",1995253664,"""2847775645""","""ILLUMINA""","""Generic""","""DMSO_0_R1 [PRO-seq]""","""SRP429713""",79590655,"""The TFIIH kinase CDK7 governs RNA polymerase II initiation, elongation, termination, and RNA processing""","""DMSO_0_R1 [PRO-seq]""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM7119231: DMSO_0_R1 [PRO-seq] Homo sapiens OTHER""","""ovarian cancer""",5805224155,"""OTHER""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM7119231""","""GSM7119231_r1""","""RNA-seq: Cells were then harvested by washing with cold D-PBS, scraped in D-PBS into 15mL conical tubes, and centrifuged at 1000rpm for 5min 4C. RNA was extracted with TRIzol according to the manufact…","""GSM7119231_r1""","""Wild-type OV90 cells""","""GSM7119231""","""2024-12-29""","""PRJNA949515""","""SRR23989399""","""SRS17155689""","""2024-12-29""","""GSE228346"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

In [6]:
# PRO-seq metadata
defined_pairs_gse228346_proseq = [
    # DMSO
    ("SRR23989395", "SRR23989399"), # DMSO_20_R1 [PRO-seq] vs DMSO_0_R1 [PRO-seq]
    ("SRR23989394", "SRR23989398"), # DMSO_20_R2 [PRO-seq] vs DMSO_0_R2 [PRO-seq]
    ("SRR23989382", "SRR23989399"), # DMSO_45_R1 [PRO-seq] vs DMSO_0_R1 [PRO-seq]
    ("SRR23989378", "SRR23989398"), # DMSO_45_R2 [PRO-seq] vs DMSO_0_R2 [PRO-seq]

    # SY5609
    ("SRR23989393", "SRR23989397"), # SY5609_20_R1 [PRO-seq] vs SY5609_0_R1 [PRO-seq]
    ("SRR23989392", "SRR23989396"), # SY5609_20_R2 [PRO-seq] vs SY5609_0_R2 [PRO-seq]
    ("SRR23989377", "SRR23989397"), # SY5609_45_R1 [PRO-seq] vs SY5609_0_R1 [PRO-seq]
    ("SRR23989376", "SRR23989396"), # SY5609_45_R2 [PRO-seq] vs SY5609_0_R2 [PRO-seq]
]


paired_metadata_gse228346_proseq = create_paired_metadata(
    GSE228346_pro_seq_combined,
    defined_pairs_gse228346_proseq,
    sample_id_column_name="accession"
)

if paired_metadata_gse228346_proseq is not None:
    paired_metadata_gse228346_proseq.write_csv(
        "../../Data/metadata/metadata_proseq/GSE228346_proseq_paired_metadata.tsv",
        separator='\t'
    )

pl.Config.set_tbl_rows(2)
display(paired_metadata_gse228346_proseq)

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,time,molecule_subtype,library_strategy,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,adapter,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy_right,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR23989395""","""SRR23989399""","""dmso_20_r1_pro-seq_ vs dmso_0_r1_pro-seq_""","""null""","""null""","""null""","""null""","""null""","""null""","""DMSO_20_R1 [PRO-seq] vs DMSO_0_R1 [PRO-seq]""","""GSM7119235 vs GSM7119231""","""Public on Dec 19 2024""","""Mar 28 2023""","""Dec 19 2024""","""SRA""","""1""","""wild type OV90 cells""","""Homo sapiens""","""9606""","""total RNA""","""PRO-seq from OV90 cells, proseq_virtual_spikein_normalized_counts.txt""","""GPL24676""","""Taylor,K,Jones""","""tajo5912@colorado.edu""","""3606246612""","""University of Colorado Boulder""","""3415 Colorado Ave""","""Boulder""","""CO""","""80301""","""USA""","""Illumina NovaSeq 6000""","""other""","""transcriptomic""","""OTHER""","""GSM7119235 vs GSM7119231""","""NONE""","""GSE228346""","""0""","""SRX19793276 vs SRX19793272""","""Wild-type OV90 cells""","""ovarian cancer""","""30min pretreatment with DMSO at 37C followed by a 42C heat shock for 20min with a 0min recovery at 37C vs 30min pretreatment with DMSO at 37C followed by a 42C heat shock for 0min with a 0min recovery…","""50 vs 30""","""Nascent RNA""","""PRO-seq""","""hg38""","""NOT PROVIDED""","""SAMN33946083 vs SAMN33946087""","""KAPA mRNA HyperPrep Kit""","""https://doi.org/10.1126/sciadv.adr9660""","""Illumina TruSeq Small RNA adapters""","""SRR23989395 vs SRR23989399""","""Illumina NovaSeq 6000 sequencing: GSM7119235: DMSO_20_R1 [PRO-seq] Homo sapiens OTHER vs Illumina NovaSeq 6000 sequencing: GSM7119231: DMSO_0_R1 [PRO-seq] Homo sapiens OTHER""","""The TFIIH kinase CDK7 governs RNA polymerase II initiation, elongation, termination, and RNA processing""","""SRA1611893""","""other""","""2024-12-29""","""1808084843 vs 1995253664""","""2668876305 vs 2847775645""","""ILLUMINA""","""Generic""","""DMSO_20_R1 [PRO-seq] vs DMSO_0_R1 [PRO-seq]""","""SRP429713""","""72094461 vs 79590655""","""The TFIIH kinase CDK7 governs RNA polymerase II initiation, elongation, termination, and RNA processing""","""DMSO_20_R1 [PRO-seq] vs DMSO_0_R1 [PRO-seq]""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM7119235: DMSO_20_R1 [PRO-seq] Homo sapiens OTHER vs Illumina NovaSeq 6000 sequencing: GSM7119231: DMSO_0_R1 [PRO-seq] Homo sapie

&nbsp;

&nbsp;

## 2. GSE173537 PRO-seq (SINGLE)

- ✅ DOWNLOAD (yone-SSD)
- ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD)
- ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD)

&nbsp;

- [12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb) で得られたデータ
- validate date: 2025-05-03
- [Characterizing primary transcriptional responses to short term heat shock in Down syndrome](https://doi.org/10.1371/journal.pone.0307375)
- 全てPRO-seqのデータで、chromosome 21 trisomic のデータも登録されている

&nbsp;

### Adapter

- About adapter: `Precision run-on sequencing experiments (PRO-seq) were carried out according to the PRO-seq protocol [Mahat DB, Salamanca HH, Duarte FM, Danko CG, Lis JT. Mammalian heat shock response and mechanisms underlying its genome-wide transcriptional regulation. Molecular cell. 2016;62(1):63–78. pmid:27052732].`
- [Mammalian heat shock response and mechanisms underlying its genome-wide transcriptional regulation](https://doi.org/10.1016/j.molcel.2016.02.025) の [Document S1. Supplemental Experimental Procedures, Figures S1–S7, and Tables S1–S7](https://www.cell.com/cms/10.1016/j.molcel.2016.02.025/attachment/a26eb1ee-03f5-423e-8282-5d6a5e331ac4/mmc1.pdf) を調べると、以下のような記述がある
- `After the third biotinstreptavidin affinity purification, cDNA was generated by reverse transcription and PRO-seq libraries were prepared for sequencing using Illumina TruSeq small-RNA adaptors with nine cycles of PCR.` とあるので、adapter は おそらく Illumina TruSeq small-RNA adaptors

In [7]:
GSE173537_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE173537/GSE173537_PEP/GSE173537_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column
    pl.lit("https://doi.org/10.1371/journal.pone.0307375").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE173537_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE173537_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE173537_iseq_filtered = GSE173537_iseq[[s.name for s in GSE173537_iseq if not (s.null_count() == GSE173537_iseq.height)]]

# display(GSE173537_PEP)

In [8]:
GSE173537_combined = GSE173537_PEP.join(
    GSE173537_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
).sort(
    "sample_geo_accession", descending=True
)

GSE173537_combined.write_csv(
    "../../Data/metadata/metadata_proseq/GSE173537_proseq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE173537_combined)

In [9]:
defined_pairs_gse173537_pro_seq = [
    ("SRR14352132", "SRR14352131"), # proseq_lymphblastoid_cells_d21_42c_rep1 (heat-stress) vs proseq_lymphblastoid_cells_d21_37c_rep1 (control)
    ("SRR14352134", "SRR14352133"), # proseq_lymphblastoid_cells_t21_42c_rep1 (heat-stress) vs proseq_lymphblastoid_cells_t21_37c_rep1 (control)
    ("SRR14352136", "SRR14352135"), # proseq_lymphblastoid_cells_d21_42c_rep2 (heat-stress) vs proseq_lymphblastoid_cells_d21_37c_rep2 (control)
    ("SRR14352138", "SRR14352137")  # proseq_lymphblastoid_cells_t21_42c_rep2 (heat-stress) vs proseq_lymphblastoid_cells_t21_37c_rep2 (control)
]

paired_metadata_df_gse173537_pro_seq = create_paired_metadata(
    combined_df=GSE173537_combined,
    defined_sample_pairs=defined_pairs_gse173537_pro_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse173537_pro_seq is not None:
    paired_metadata_df_gse173537_pro_seq.write_csv(
        "../../Data/metadata/metadata_proseq/GSE173537_proseq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse173537_pro_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_type,genotype,treatment,library_strategy,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,adapter,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy_right,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR14352132""","""SRR14352131""","""proseq_lymphblastoid_cells_d21_42c_rep1 vs proseq_lymphblastoid_cells_d21_37c_rep1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR14352132 vs SRR14352131""","""SRX10705979 vs SRX10705978""","""PROseq_Lymphblastoid cells_D21_42C_rep1 vs PROseq_Lymphblastoid cells_D21_37C_rep1""","""GSM5269438 vs GSM5269437""","""Public on Apr 21 2023""","""Apr 28 2021""","""Apr 21 2023""","""SRA""","""1""","""lymphoblastoid-EBV immortalized""","""Homo sapiens""","""9606""","""total RNA""","""Eric_D21, nascent RNA (PRO-seq)""","""GPL18573""","""Mary,Ann,Allen""","""mary.a.allen@colorado.edu""","""608-209-6921""","""Allen""","""BioFrontiers""","""University of Colorado""","""UCB 596 JSCBB""","""Boulder""","""Co""","""80309""","""USA""","""Illumina NextSeq 500""","""other""","""transcriptomic""","""OTHER""","""GSM5269438 vs GSM5269437""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5269nnn/GSM5269438/suppl/GSM5269438_EricHS_S4_R1_001.bedGraph.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5269nnn/GSM5269437/suppl/GSM5269437_EricControl_…","""GSE173537""","""0""","""SRX10705979 vs SRX10705978""","""lymphoblastoid""","""disomic""","""42C 1 hour heat shock vs untreated (37C incubator)""","""PRO-seq""","""hg38 (GRCh38)""","""bedgraphs supplied for PRO-seq libraries.""","""SAMN18915532 vs SAMN18915533""","""unknown""","""https://doi.org/10.1371/journal.pone.0307375""","""Illumina TruSeq Small RNA adapters""","""SRR14352132 vs SRR14352131""","""NextSeq 500 sequencing: GSM5269438: PROseq_Lymphblastoid cells_D21_42C_rep1 Homo sapiens OTHER vs NextSeq 500 sequencing: GSM5269437: PROseq_Lymphblastoid cells_D21_37C_rep1 Homo sapiens OTHER""","""Blood cells with trisomy 21 overreact to heat shock stress [PRO-seq]""","""SRA1225197""","""other""","""2023-05-06""","""2148549019 vs 2036005632""","""3303877376 vs 3129483502""","""ILLUMINA""","""Generic""","""PROseq_Lymphblastoid cells_D21_42C_rep1 vs PROseq_Lymphblastoid cells_D21_37C_rep1""","""SRP316802""","""112213757 vs 106351450""","""Blood cells with trisomy 21 overreact to heat shock stress [PRO-seq]""","""PROseq_Lymphblastoid cells_D21_42C_rep1 vs PROseq_Lymphblastoid cells_D21_37C_rep1

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. GSE211651 PRO-seq (SINGLE)

- ✅ DOWNLOAD (yone-SSD)
- ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD)
- ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD)

&nbsp;

- validation date: 2025-05-03
- [Enhancing HIV-1 latency reversal through regulating the elongating RNA Pol II pause-release by a small-molecule disruptor of PAF1C](https://doi.org/10.1126/sciadv.adf2468)
- きちんと熱ストレス条件とコントロール条件があるデータ
- `inhibitor of PAF1C (iPAF1C)` や `shRNA` 処理も重ねて行われているデータも
- **RNA-seqのデータを見ると、熱ストレスに曝露されているデータは含まれていなかったので、PRO-seqのみ念の為収集しておく**
- [12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動

&nbsp;

### Adapter

- About adapter: 以下のような記述がある
- `PRO-seq was performed according to the previously published protocol with minor modifications (80 D. B. Mahat, H. Kwak, G. T. Booth, I. H. Jonkers, C. G. Danko, R. K. Patel, C. T. Waters, K. Munson, L. J. Core, J. T. Lis, Base-pair-resolution genome-wide mapping of active RNA polymerases using precision nuclear run-on (PRO-seq). Nat. Protoc. 11, 1455–1476 (2016).).`
- [Base-pair-resolution genome-wide mapping of active RNA polymerases using precision nuclear run-on (PRO-seq)](https://doi.org/10.1038/nprot.2016.086)
- この論文には、以下のような記載がある
- `148 Pre-process the raw sequence data. Filter out low-quality reads and trim potential adaptor sequences (TGGAATTCTCGGGTGCCAAGG) from the sequence reads. Tools such as 'cutadapt' are publicly available for this purpose. Depending on the quality of the library, sequences containing only the adaptor sequences (adaptor dimers) may take up to 5% of total reads.`
- この adapter sequence `TGGAATTCTCGGGTGCCAAGG` は、[TruSeq Small RNA](https://support-docs.illumina.com/SHARE/AdapterSequences/Content/TruSeq-SmallRNA.htm)

In [10]:
GSE211651_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE211651/GSE211651_PEP/GSE211651_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "RNA-Seq") | (pl.col("sample_title").str.contains(r'PRO-seq_'))
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("NEBNext Ultra II Directional RNA Library Prep Kit").alias("rna_library"), # add rna_library column from GEO webpage (NEW ENGLAND Biolabs)
    pl.lit("https://doi.org/10.1126/sciadv.adf2468").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# PRO-seq
GSE211651_PEP_pro_seq = GSE211651_PEP.filter(
    pl.col("sample_title").str.contains(r'PRO-seq_')
).with_columns(
    pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")
)

# metadata from iseq
GSE211651_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE211651_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE211651_iseq_filtered = GSE211651_iseq[[s.name for s in GSE211651_iseq if not (s.null_count() == GSE211651_iseq.height)]]

# display(GSE211651_PEP_pro_seq)

In [11]:
GSE211651_combined_pro_seq = GSE211651_PEP_pro_seq.join(
    GSE211651_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE211651_combined_pro_seq.write_csv(
    "../../Data/metadata/metadata_proseq/GSE211651_proseq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE211651_combined_pro_seq)

In [12]:
# PRO-seq
defined_pairs_gse211651_pro_seq = [
    # PAF1 knockdown
    ("SRR21140707", "SRR21140711"), # PRO-seq_DLD1_shPAF1_HS vs PRO-seq_DLD1_shPAF1
    ("SRR21140708", "SRR21140712"), # PRO-seq_DLD1_shPAF1_HS vs PRO-seq_DLD1_shPAF1
    ("SRR21140709", "SRR21140713"), # PRO-seq_DLD1_shPAF1_HS vs PRO-seq_DLD1_shPAF1
    ("SRR21140710", "SRR21140714"), # PRO-seq_DLD1_shPAF1_HS vs PRO-seq_DLD1_shPAF1
    # WT (DMSO)
    ("SRR21140715", "SRR21140719"), # PRO-seq_DLD1_DMSO_HS vs PRO-seq_DLD1_DMSO
    ("SRR21140716", "SRR21140720"), # PRO-seq_DLD1_DMSO_HS vs PRO-seq_DLD1_DMSO
    ("SRR21140717", "SRR21140721"), # PRO-seq_DLD1_DMSO_HS vs PRO-seq_DLD1_DMSO
    ("SRR21140718", "SRR21140722"), # PRO-seq_DLD1_DMSO_HS vs PRO-seq_DLD1_DMSO
    # WT (iPAF1C 20uM)
    ("SRR21140723", "SRR21140727"), # PRO-seq_DLD1_iPAF1C_HS vs PRO-seq_DLD1_iPAF1C
    ("SRR21140724", "SRR21140728"), # PRO-seq_DLD1_iPAF1C_HS vs PRO-seq_DLD1_iPAF1C
    ("SRR21140725", "SRR21140729"), # PRO-seq_DLD1_iPAF1C_HS vs PRO-seq_DLD1_iPAF1C
    ("SRR21140726", "SRR21140730")  # PRO-seq_DLD1_iPAF1C_HS vs PRO-seq_DLD1_iPAF1C
]

paired_metadata_df_gse211651_pro_seq = create_paired_metadata(
    combined_df=GSE211651_combined_pro_seq,
    defined_sample_pairs=defined_pairs_gse211651_pro_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse211651_pro_seq is not None:
    paired_metadata_df_gse211651_pro_seq.write_csv(
        "../../Data/metadata/metadata_proseq/GSE211651_proseq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse211651_pro_seq)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,tissue,cell_line,cell_type,genotype,treatment,chip_antibody,assembly,sample_accession,rna_library,reference_doi,adapter,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR21140707""","""SRR21140711""","""pro-seq_dld1_shpaf1_hs vs pro-seq_dld1_shpaf1""","""null""","""null""","""null""","""null""","""null""","""null""","""PRO-seq_DLD1_shPAF1_HS vs PRO-seq_DLD1_shPAF1""","""GSM6482063 vs GSM6482062""","""Public on Mar 23 2023""","""Aug 19 2022""","""Mar 23 2023""","""SRA""","""1""","""colon""","""Homo sapiens""","""9606""","""total RNA""","""GPL18573""","""Ali,,Shilatifard""","""ash@northwestern.edu""","""Shilatifard Lab""","""Department of Biochemistry and Molecular Genetics""","""320 E Superior St""","""Chicago""","""IL""","""60611""","""USA""","""Illumina NextSeq 500""","""other""","""transcriptomic""","""OTHER""","""GSM6482063 vs GSM6482062""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6482nnn/GSM6482063/suppl/GSM6482063_PRO-seq_DLD1_shPAF1_HS.minus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6482nnn/GSM6482062/suppl/GSM6482062_PRO-seq_D…","""GSE211651""","""0""","""SRX17152994 vs SRX17152993""","""colon""","""DLD-1""","""colorectal adenocarcinoma""","""PAF1 knockdown""","""heat shock vs untreated""","""null""","""hg38""","""SAMN30413337 vs SAMN30413338""","""NEBNext Ultra II Directional RNA Library Prep Kit""","""https://doi.org/10.1126/sciadv.adf2468""","""Illumina TruSeq Small RNA adapters""","""SRR21140707 vs SRR21140711""","""NextSeq 500 sequencing: GSM6482063: PRO-seq_DLD1_shPAF1_HS Homo sapiens OTHER vs NextSeq 500 sequencing: GSM6482062: PRO-seq_DLD1_shPAF1 Homo sapiens OTHER""","""Enhancing HIV-1 latency reversal through regulating the elongating RNA Pol II pause-release by a small-molecule disruptor of PAF1C""","""SRA1479137""","""other""","""2023-03-26""","""224707969 vs 133473142""","""348179135 vs 212104138""","""ILLUMINA""","""Generic""","""PRO-seq_DLD1_shPAF1_HS vs PRO-seq_DLD1_shPAF1""","""SRP393054""","""16064642 vs 9497152""","""Enhancing HIV-1 latency reversal through regulating the elongating RNA Pol II pause-release by a small-molecule disruptor of PAF1C""","""PRO-seq_DLD1_shPAF1_HS vs PRO-seq_DLD1_shPAF1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM6482063: PRO-seq_DLD1_shPAF1_HS Homo sapiens OTHER vs NextSeq 500 sequencing: GSM6482062: PRO-seq_DLD1_shPAF1 Homo sapiens OTHER""","""colorectal adenocarcinoma""","""812775291 vs 480559228""","""OTHER""","""TRANSCRIPTOM

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. GSE154746 PRO-seq (PAIRED)

- ✅ DOWNLOAD (yone-SSD)
- ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD)
- ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD)

&nbsp;

- [12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から移行
- validate date: 2025-05-04
- [Stress-induced transcriptional memory accelerates promoter-proximal pause release and decelerates termination over mitotic divisions](https://doi.org/10.1016/j.molcel.2021.03.007)
- [実験医学: PRO-seq](https://www.yodosha.co.jp/jikkenigaku/keyword/PRO-seq/id/107099)
- `unconditioned (uC) or preconditioned with a single heat shock exposure (pC)` という条件があるが、これらの条件は外すことに決定
- 42℃で複数の曝露時間条件が含まれている

&nbsp;

### Adapter

- 以下のような記述がある
- `The libraries were generated using **TruSeq small-RNA adaptors** and sequenced using NextSeq500 (Illumina).`

In [13]:
ID = "GSE154746"

GSE154746_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq small-RNA adaptor").alias("rna_library"), # add rna_library column from GEO webpage (Illumina)
    pl.lit("https://doi.org/10.1016/j.molcel.2021.03.007").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE154746_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE154746_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE154746_iseq_filtered = GSE154746_iseq[[s.name for s in GSE154746_iseq if not (s.null_count() == GSE154746_iseq.height)]]

# display(GSE154746_PEP)

In [14]:
GSE154746_combined_pro_seq = GSE154746_PEP.join(
    GSE154746_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE154746_combined_pro_seq.write_csv(
    "../../Data/metadata/metadata_proseq/GSE154746_proseq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE154746_combined_pro_seq)

In [15]:
defined_pairs_gse154746_pro_seq = [
    ("SRR12267709", "SRR12267707"), # proseq_k562_s_uc12p5_rep1 vs proseq_k562_s_uc0_rep1
    ("SRR12267710", "SRR12267708"), # PROseq_K562_s_uC12p5_rep2 vs PROseq_K562_s_uC0_rep2
    ("SRR12267711", "SRR12267707"), # PROseq_K562_s_uC25_rep1 vs PROseq_K562_s_uC0_rep1
    ("SRR12267712", "SRR12267708"), # PROseq_K562_s_uC25_rep2 vs PROseq_K562_s_uC0_rep2
    ("SRR12267713", "SRR12267707"), # PROseq_K562_s_uC40_rep1 vs PROseq_K562_s_uC0_rep1
    ("SRR12267714", "SRR12267708"), # PROseq_K562_s_uC40_rep2 vs PROseq_K562_s_uC0_rep2
    # ("SRR12267717", "SRR12267715"), # PROseq_K562_s_pC12p5_rep1 vs PROseq_K562_s_pC0_rep1
    # ("SRR12267718", "SRR12267716"), # PROseq_K562_s_pC12p5_rep2 vs PROseq_K562_s_pC0_rep2
    # ("SRR12267719", "SRR12267715"), # PROseq_K562_s_pC25_rep1 vs PROseq_K562_s_pC0_rep1
    # ("SRR12267720", "SRR12267716"), # PROseq_K562_s_pC25_rep2 vs PROseq_K562_s_pC0_rep2
    # ("SRR12267721", "SRR12267715"), # PROseq_K562_s_pC40_rep1 vs PROseq_K562_s_pC0_rep1
    # ("SRR12267722", "SRR12267716")  # PROseq_K562_s_pC40_rep2 vs PROseq_K562_s_pC0_rep2
]

paired_metadata_df_gse154746_pro_seq = create_paired_metadata(
    combined_df=GSE154746_combined_pro_seq,
    defined_sample_pairs=defined_pairs_gse154746_pro_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse154746_pro_seq is not None:
    paired_metadata_df_gse154746_pro_seq.write_csv(
        "../../Data/metadata/metadata_proseq/GSE154746_proseq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse154746_pro_seq)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_characteristics_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,preconditioning,sequenced_molecule,library_strategy,genome_build,sample_accession,rna_library,reference_doi,adapter,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy_right,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR12267709""","""SRR12267707""","""proseq_k562_s_uc12p5_rep1 vs proseq_k562_s_uc0_rep1""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR12267709 vs SRR12267707""","""SRX8772768 vs SRX8772766""","""PROseq_K562_s_uC12p5_rep1 vs PROseq_K562_s_uC0_rep1""","""GSM4678147 vs GSM4678145""","""Public on Apr 01 2021""","""Jul 20 2020""","""Apr 01 2021""","""SRA""","""1""","""K562""","""Homo sapiens""","""9606""","""heat shock (additional to preconditioning): 12.5 min at 42°C water bath (initiated with prewarmed preconditioned media) vs heat shock (additional to preconditioning): no heat stress applied""","""total RNA""","""3' barcode:, 3' UMI:, 6nt + C (7 nt in total), 5' UMI:, 6nt + C (7 nt in total), Normalized_PROseq_K562_s_uC12p5_BRs_mn.bigWig, Normalized_PROseq_K562_s_uC12p5_BRs_pl.bigWig vs 3' barcode:, 3' UMI:, 6…","""GPL16791""","""Anniina,,Vihervaara""","""mavih@kth.se""","""+358 50 5463694""","""Vihervaara""","""Gene Technology""","""KTH Royal Institute of Technology""","""Tomtebodavägen 23A""","""Stockholm""","""17165""","""Sweden""","""Illumina HiSeq 2500""","""other""","""transcriptomic""","""OTHER""","""GSM4678147 vs GSM4678145""","""NONE""","""GSE154746""","""0""","""SRX8772768 vs SRX8772766""","""K562""","""none""","""nascent RNA""","""PRO-seq""","""hg19""","""SAMN15582320 vs SAMN15582322""","""TruSeq small-RNA adaptor""","""https://doi.org/10.1016/j.molcel.2021.03.007""","""Illumina TruSeq Small RNA adapters""","""SRR12267709 vs SRR12267707""","""Illumina HiSeq 2500 sequencing: GSM4678147: PROseq_K562_s_uC12p5_rep1 Homo sapiens OTHER vs Illumina HiSeq 2500 sequencing: GSM4678145: PROseq_K562_s_uC0_rep1 Homo sapiens OTHER""","""Heat-induced transcriptional memory profiled in human K562 erythroleukemia cells [PRO-seq]""","""SRA1100915""","""other""","""2021-04-02""","""2692118686 vs 3187245265""","""1435699311;1530256531 vs 1663974603;1788871689""","""ILLUMINA""","""Generic""","""PROseq_K562_s_uC12p5_rep1 vs PROseq_K562_s_uC0_rep1""","""SRP272608""","""23952228 vs 28133239""","""Heat-induced transcriptional memory profiled in human K562 erythroleukemia cells [PRO-seq]""","""PROseq_K562_s_uC12p5_rep1 vs PROseq_K562_s_uC0_rep1""","""Illumina HiSeq 2500""","""Ill

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 5. GSE202187 PRO-seq (SINGLE) 

- ✅ DOWNLOAD (yone-SSD)
- ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD)
- ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD)

&nbsp;

- [12_4_metadata_modified_false_no_control.ipynb](./12_4_metadata_modified_false_no_control.ipynb)から移動
- validation date: 2025-05-06
- [SPT6 functions in transcriptional pause/release via PAF1C recruitment](https://doi.org/10.1016/j.molcel.2022.06.037)
- GSE202189に関連するデータで、PRO-seqのデータ
- PRO-seqとSPT6に対するauxin-inducible degradation (AID) systemが行われている論文
- 42℃, 60min. 

&nbsp;

### Adapter

- 以下のような記述が論文にある
- `Low-quality bases and adapters from 3’ ends were removed using cutadapt 1.14 (Martin, 2011) requiring a read length of 16–36 bp, followed by removal of reads that were derived from ribosomal RNA.`
- Gene Expression Omnibus のページ、例えば [GSM6102924](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM6102924) の Extraction protocol には以下のような記載がある
- `For PRO-seq, adapter sequences previously described (Mahat et al., 2016) were used.`
- したがって、`Illumina TruSeq Small RNA adapters` が妥当であると考えられる

In [16]:
GSE202187_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE202187/GSE202187_PEP/GSE202187_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Illumina TruSeq small RNA library prep kit").alias("rna_library"), # add rna_library column from GEO webpage (Illumina)
    pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE202187 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE202187_metadata.tsv",
    separator='\t'
)

GSE202187_iseq_filtered = GSE202187[[s.name for s in GSE202187 if not (s.null_count() == GSE202187.height)]]

# display(GSE202187_PEP)

In [17]:
GSE202187_combined = GSE202187_PEP.join(
    GSE202187_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE202187_combined.write_csv(
    "../../Data/metadata/metadata_proseq/GSE202187_proseq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE202187_combined)

In [18]:
defined_pairs_gse202187 = [
    ("SRR19077094", "SRR19077096"), # PRO-seq-01-SPT6AID-Veh2h-HS1h vs PRO-seq-01-SPT6AID-Veh2h-NHS1h
    ("SRR19077093", "SRR19077095"), # PRO-seq-01-SPT6AID-Aux2h-HS1h vs PRO-seq-01-SPT6AID-Aux2h-NHS1h
    ("SRR19077090", "SRR19077092"), # PRO-seq-02-SPT6AID-Veh2h-HS1h vs PRO-seq-02-SPT6AID-Veh2h-NHS1h
    ("SRR19077089", "SRR19077091"), # PRO-seq-02-SPT6AID-Aux2h-HS1h vs PRO-seq-02-SPT6AID-Aux2h-NHS1h
    ("SRR19077086", "SRR19077088"), # PRO-seq-03-PAF1AID-Veh2h-HS1h vs PRO-seq-03-PAF1AID-Veh2h-NHS1h
    ("SRR19077085", "SRR19077087"), # PRO-seq-03-PAF1AID-Aux2h-HS1h vs PRO-seq-03-PAF1AID-Aux2h-NHS1h
    ("SRR19077082", "SRR19077084"), # PRO-seq-04-PAF1AID-Veh2h-HS1h vs PRO-seq-04-PAF1AID-Veh2h-NHS1h
    ("SRR19077081", "SRR19077083")  # PRO-seq-04-PAF1AID-Aux2h-HS1h vs PRO-seq-04-PAF1AID-Aux2h-NHS1h
]

paired_metadata_df_gse202187 = create_paired_metadata(
    combined_df=GSE202187_combined, 
    defined_sample_pairs=defined_pairs_gse202187,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse202187 is not None:
    paired_metadata_df_gse202187.write_csv(
        "../../Data/metadata/metadata_proseq/GSE202187_proseq_paired_metadata.tsv",
        separator='\t'
    )

display(paired_metadata_df_gse202187)

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,assembly,library_strategy,sample_accession,rna_library,adapter,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_aspera,fastq_galaxy,sample_description_right,sra_galaxy,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,cell_type_right,fastq_ftp,base_count,library_strategy_right,library_source,sra_ftp,library_layout,status,fastq_aspera,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR19077094""","""SRR19077096""","""pro-seq-01-spt6aid-veh2h-hs1h vs pro-seq-01-spt6aid-veh2h-nhs1h""","""null""","""null""","""null""","""null""","""null""","""null""","""PRO-seq-01-SPT6AID-Veh2h-HS1h vs PRO-seq-01-SPT6AID-Veh2h-NHS1h""","""GSM6102926 vs GSM6102924""","""Public on Aug 09 2022""","""May 04 2022""","""Aug 09 2022""","""SRA""","""1""","""DLD-1""","""Homo sapiens""","""9606""","""nuclear RNA""","""PRO-seq""","""GPL24676""","""Yuki,,Aoi""","""yuki.aoi@northwestern.edu""","""Ali Shilatifard""","""Biochemistry & Molecular Genetics""","""Northwestern University""","""303 E Superior St. SQBRC 7-300""","""Chicago""","""IL""","""60611""","""USA""","""Illumina NovaSeq 6000""","""other""","""transcriptomic""","""OTHER""","""GSM6102926 vs GSM6102924""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102926/suppl/GSM6102926_PRO-seq-01-SPT6AID-Veh2h-HS1h.minus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102924/suppl/GSM6102924_PR…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102926/suppl/GSM6102926_PRO-seq-01-SPT6AID-Veh2h-HS1h.plus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102924/suppl/GSM6102924_PRO…","""GSE202187, GSE202190""","""0""","""SRX15147078 vs SRX15147076""","""DLD-1""","""Colorectal adenocarcinoma""","""SPT6-AID OsTIR1""","""Vehcle 2h HS 1h vs Vehcle 2h NHS 1h""","""hg38""","""PRO-seq""","""SAMN28088070 vs SAMN28088072""","""Illumina TruSeq small RNA library prep kit""","""Illumina TruSeq Small RNA adapters""","""SRR19077094 vs SRR19077096""","""Illumina NovaSeq 6000 sequencing: GSM6102926: PRO-seq-01-SPT6AID-Veh2h-HS1h Homo sapiens OTHER vs Illumina NovaSeq 6000 sequencing: GSM6102924: PRO-seq-01-SPT6AID-Veh2h-NHS1h Homo sapiens OTHER""","""SPT6 functions in transcriptional pause-release via PAF1C recruitment [PRO-seq]""","""SRA1414556""","""other""","""2022-08-24""","""1657072733 vs 1500066373""","""2342367318 vs 2108794344""","""ILLUMINA""","""fasp.sra.ebi.ac.uk:/vol1/srr/SRR190/094/SRR19077094 vs fasp.sra.ebi.ac.uk:/vol1/srr/SRR190/096/SRR19077096""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR190/094/SRR19077094/SRR19077094.fastq

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 6. GSE127844 PRO-seq (SINGLE) 

- ✅ DOWNLOAD (yone-SSD)
- ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD)
- ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD)

&nbsp;

- validate date: 2025-05-03
- [Stress-induced transcriptional memory accelerates promoter-proximal pause release and decelerates termination over mitotic divisions](https://doi.org/10.1016/j.molcel.2021.03.007)
- 以前の研究で使用したPRO-seqのデータ
- HSF1のsilencing 処理も含まれている
- コントロールデータが含まれているので念の為収集
- [12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動

&nbsp;

### Adapter

- 以下のような記述が論文内にある
- `The libraries were generated using TruSeq small-RNA adaptors and sequenced using NextSeq500 (Illumina).`

In [19]:
GSE127844_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE127844/GSE127844_PEP/GSE127844_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column
    pl.lit("https://doi.org/10.1016/j.molcel.2021.03.007").alias("reference_doi"),  # add DOI of the reference paper column
    pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE127844_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE127844_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE127844_iseq_filtered = GSE127844_iseq[[s.name for s in GSE127844_iseq if not (s.null_count() == GSE127844_iseq.height)]]

# display(GSE127844_PEP)

In [20]:
GSE127844_combined = GSE127844_PEP.join(
    GSE127844_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE127844_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE127844_pro_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE127844_combined)

In [21]:
defined_pairs_gse127844_pro_seq = [
    ("SRR8669164", "SRR8669162"), # PROseq_unCond_K562_scr_HS30_rep1 vs PROseq_unCond_K562_scr_HS0_rep1
    ("SRR8669165", "SRR8669163"), # PROseq_unCond_K562_scr_HS30_rep2 vs PROseq_unCond_K562_scr_HS0_rep2
    ("SRR8669166", "SRR8669162"), # PROseq_unCond_K562_scr_HS60_rep1 vs PROseq_unCond_K562_scr_HS0_rep1
    ("SRR8669169", "SRR8669167"), # PROseq_preCond_K562_scr_HS30_rep1 vs PROseq_preCond_K562_scr_HS0_rep1
    ("SRR8669170", "SRR8669168"), # PROseq_preCond_K562_scr_HS30_rep2 vs PROseq_preCond_K562_scr_HS0_rep2
    ("SRR8669171", "SRR8669167"), # PROseq_preCond_K562_scr_HS60_rep1 vs PROseq_preCond_K562_scr_HS0_rep1
    ("SRR8669173", "SRR8669172"), # PROseq_unCond_K562_shHSF1_HS30_rep1 vs PROseq_unCond_K562_shHSF1_HS0_rep1
    ("SRR8669174", "SRR8669172"), # PROseq_unCond_K562_shHSF1_HS60_rep1 vs PROseq_unCond_K562_shHSF1_HS0_rep1
    ("SRR8669176", "SRR8669175"), # PROseq_preCond_K562_shHSF1_HS30_rep1 vs PROseq_preCond_K562_shHSF1_HS0_rep1
    ("SRR8669177", "SRR8669175")  # PROseq_preCond_K562_shHSF1_HS60_rep1 vs PROseq_preCond_K562_shHSF1_HS0_rep1
]

paired_metadata_df_gse127844_pro_seq = create_paired_metadata(
    combined_df=GSE127844_combined,
    defined_sample_pairs=defined_pairs_gse127844_pro_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse127844_pro_seq is not None:
    paired_metadata_df_gse127844_pro_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE127844_pro_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse127844_pro_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_characteristics_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,hsf1_depletion,preconditioning,genome_build,reanalyzed_by,sample_accession,rna_library,reference_doi,adapter,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR8669164""","""SRR8669162""","""proseq_uncond_k562_scr_hs30_rep1 vs proseq_uncond_k562_scr_hs0_rep1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR8669164 vs SRR8669162""","""SRX5465138 vs SRX5465136""","""PROseq_unCond_K562_scr_HS30_rep1 vs PROseq_unCond_K562_scr_HS0_rep1""","""GSM3639852 vs GSM3639850""","""Public on Mar 04 2020""","""Mar 05 2019""","""Mar 04 2020 vs Jul 29 2021""","""SRA""","""1""","""K562""","""Homo sapiens""","""9606""","""heat shock (additional to preconditioning): 30 min at 42°C water bath vs heat shock (additional to preconditioning): no heat stress applied""","""total RNA""","""nascent RNA, Normalized_PROseq_unCond_K562_scr_HS30_minusStrand.bigWig, Normalized_PROseq_unCond_K562_scr_HS30_plusStrand.bigWig vs nascent RNA, Normalized_PROseq_unCond_K562_scr_HS0_minusStrand.bigWi…","""GPL18573""","""Anniina,,Vihervaara""","""mavih@kth.se""","""+358 50 5463694""","""Vihervaara""","""Gene Technology""","""KTH Royal Institute of Technology""","""Tomtebodavägen 23A""","""Stockholm""","""17165""","""Sweden""","""Illumina NextSeq 500""","""other""","""transcriptomic""","""OTHER""","""GSM3639852 vs GSM3639850""","""NONE""","""GSE127844""","""0""","""SRX5465138 vs SRX5465136""","""no depletion (scrambled sequence transfected)""","""none""","""hg19""",null,"""SAMN11055121 vs SAMN11055123""","""unknown""","""https://doi.org/10.1016/j.molcel.2021.03.007""","""Illumina TruSeq Small RNA adapters""","""SRR8669164 vs SRR8669162""","""NextSeq 500 sequencing: GSM3639852: PROseq_unCond_K562_scr_HS30_rep1 Homo sapiens OTHER vs NextSeq 500 sequencing: GSM3639850: PROseq_unCond_K562_scr_HS0_rep1 Homo sapiens OTHER""","""Heat-induced transcriptional memory profiled in human K562 erythroleukemia cells.""","""SRA856092""","""other""","""2020-03-05""","""830749447 vs 763923167""","""967160325 vs 878641086""","""ILLUMINA""","""Generic""","""PROseq_unCond_K562_scr_HS30_rep1 vs PROseq_unCond_K562_scr_HS0_rep1""","""SRP187541""","""30157389 vs 27689611""","""Heat-induced transcriptional memory profiled in human K562 erythroleukemia cells.""","""PROseq_unCond_K562_scr_HS30_rep1 vs PROseq_unCond_K562_scr_HS0_rep1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM3639852: PROseq_unCond_K562_scr_HS30_rep1 Homo sapiens 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# Add Another PRO-seq data (or GRO-seq)

- これまでは、RNA-seqを含むプロジェクトに付随するPRO-seqのデータを処理してきた
- ここからは、新しく以下のクエリでGene Expression Omnibus (GEO)から取得してきたデータのメタデータを解析する
- `#12,"Search ((((""homo sapiens""[Organism]) AND ""other""[DataSet Type]) AND ""gds sra""[Filter])) AND ((((((""heat shock response""[MeSH Terms]) OR ""heat shock""[Description]) OR ""heat stress""[Description]) OR ""high temperature stress""[Description]) OR ""hyperthermia""[Description]) OR ""heat stroke""[Description])",45,02:52:10`

&nbsp;

- [../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified.md](../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified.md)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 7. GSE66448 (SINGLE) GRO-seq

- ✅ DOWNLOAD (yone-SSD)
<!-- - ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD) -->
<!-- - ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD) -->

&nbsp;

- validate date: 2026-02-21
- [Global SUMOylation on active chromatin is an acute heat stress response restricting transcription](https://doi.org/10.1186/s13059-015-0717-y)
- [GRO-Seq/BRIC-Seq/Bru-Seq/BruChase-Seq](https://jp.illumina.com/science/sequencing-method-explorer/kits-and-arrays/gro-seq-bric-seq-bru-seq-bruchase-seq.html)
- [Global Run-On sequencing (GRO-seq)](https://doi.org/10.1007/978-1-4939-4035-6_9)

&nbsp;

### Adapter

- PRO-seq と手法が異なるので注意
- [GSM1622614](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1622614) などを参考にする
- `Poly(A)-tailing was followed by cDNA synthesis using complementary poly(T)-primers.` と書いてある
- [Global Run-On sequencing (GRO-seq)](https://doi.org/10.1007/978-1-4939-4035-6_9) などを見ると、`NTI223 - /5Phos/GAT CGT CGG ACT GTA GAA CTC T/idSp/CA AGC AGA AGA CGG CAT ACG ATT TTT TTT TTT TTT TTT TTT VN` といった配列が記載されている
- [proseq2.0](https://github.com/Danko-Lab/proseq2.0)を使用する前に、poly(A) を除去するステップを噛ませる必要がある

In [22]:
GSE66448_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE66448/GSE66448_PEP/GSE66448_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("organism") == "Homo sapiens") &
    (pl.col("protocol") != "ChIP")
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    (pl.lit("unknown").alias("rna_library")),
    (pl.lit("https://doi.org/10.1186/s13059-015-0717-y").alias("reference_doi")),
    (pl.lit("43 C").alias("stress treatment temperature")),
    (pl.lit("30 min.").alias("stress treatment time")),
    (pl.lit("37 C").alias("control treatment temperature"))
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE66448_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE66448.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE66448_iseq_filtered = GSE66448_iseq[[s.name for s in GSE66448_iseq if not (s.null_count() == GSE66448_iseq.height)]]

# display(GSE66448_PEP)

In [23]:
GSE66448_combined = GSE66448_PEP.join(
    GSE66448_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
).sort(
    "sample_geo_accession", descending=True
)

GSE66448_combined.write_csv(
    "../../Data/metadata/metadata_proseq/GSE66448_groseq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE66448_combined)

In [24]:
defined_pairs_gse66448 = [
    ("SRR1823903", "SRR1823901"), # K562-GROseq-HS-rep1 vs K562-GROseq-C-rep1
    ("SRR1823904", "SRR1823902")  # K562-GROseq-Hs-rep2 vs K562-GROseq-C-rep2
]

paired_metadata_df_gse66448 = create_paired_metadata(
    combined_df=GSE66448_combined,
    defined_sample_pairs=defined_pairs_gse66448,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse66448 is not None:
    paired_metadata_df_gse66448.write_csv(
        "../../Data/metadata/metadata_proseq/GSE66448_groseq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse66448)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,treatment1,chip_antibody,processed_data_file,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR1823903""","""SRR1823901""","""k562-groseq-hs-rep1 vs k562-groseq-c-rep1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR1823903 vs SRR1823901""","""SRX895425 vs SRX895423""","""K562-GROseq-HS-rep1 vs K562-GROseq-C-rep1""","""GSM1622614 vs GSM1622612""","""Public on Jul 02 2015""","""Mar 03 2015""","""May 15 2019""","""SRA""","""1""","""K562 cells""","""Homo sapiens""","""9606""","""Heat shock for GRO-seq was done in 43 degree water bath for 30min. Recovery was done by incubating cells at 37 degree C for 1h after heat shock.""","""K562 cells were grown in RPMI1640 medium supplemented with 10% fetal bovine serum, 1% penicillin/streptomycin antibiotics and 2mM l-glutamine at 37 degree C temperature incubator with 5% CO2.""","""total RNA""","""Nuclei were extracted from 5-10 million cells and nuclear run-on reactions were performed for 5 minutes at 30°C in the presence of Sarkosyl and BrUTP. RNA was extracted with Trizol, DNase-treated, bas…","""nascent RNA, GRO-seq""","""GPL11154""","""Einari,A,Niskanen""","""einari.niskanen@uef.fi""","""Biomedicine""","""School of Medicine""","""University of Eastern Finland""","""Yliopistonranta 8""","""Kuopio""","""70210""","""Finland""","""Illumina HiSeq 2000""","""other""","""transcriptomic""","""OTHER""","""GSM1622614 vs GSM1622612""","""NONE""","""GSE66448""","""0""","""SRX895425 vs SRX895423""","""K562""","""heat shock (30min at 43 degree C) vs control (37 degree C)""","""null""","""K562-GROseq-HS-plus.bedGraph.gz; K562-GROseq-HS-minus.bedGraph.gz vs K562-GROseq-C-plus.bedGraph.gz; K562-GROseq-C-minus.bedGraph.gz""","""hg19""","""ChIP-seq and GRO-seq signal intensities as bedGraph files.""","""SAMN03383046 vs SAMN03383044""","""unknown""","""https://doi.org/10.1186/s13059-015-0717-y""","""43 C""","""30 min.""","""37 C""","""SRR1823903 vs SRR1823901""","""Illumina HiSeq 2000 sequencing: GSM1622614: K562-GROseq-HS-rep1 Homo sapiens OTHER vs Illumina HiSeq 2000 sequencing: GSM1622612: K562-GROseq-C-rep1 Homo sapiens OTHER""","""Homo

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 8. [GSE89230](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE89230) (SINGLE) PRO-seq

- ✅ DOWNLOAD (yone-SSD)
- ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD)
- ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD)

&nbsp;

- validate date: 2026-02-21
- [Transcriptional response to stress is pre-wired by promoter and enhancer architecture](https://doi.org/10.1038/s41467-017-00151-0)
- 42℃ 30 minutes で K562 での研究
- コントロールも存在するので収集する

&nbsp;

### Adapter

- Gene Expression Omnibus の [GSE89230](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE89230) のページ (e.g., [GSM2361442](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2361442)) を見ると、以下のような記載が `Extraction protocol` に書いてある
- `The libraries were generated using TruSeq small-RNA adaptors, amplified 11 cycles, gel-purified and size-selected to 140-350 bp fragments. The libraries were sequenced using HiSeq 2000 (Illumina).`

In [25]:
GSE89230_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE89230/GSE89230_PEP/GSE89230_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Illumina TruSeq Small RNA Library Preparation Kits").alias("rna_library"), # add rna_library column from article (Illumina)
    pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE89230 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE89230_metadata.tsv",
    separator='\t'
)

GSE89230_iseq_filtered = GSE89230[[s.name for s in GSE89230 if not (s.null_count() == GSE89230.height)]]

# pl.Config.set_tbl_rows(4)
# display(GSE89230_PEP)

In [26]:
GSE89230_combined = GSE89230_PEP.join(
    GSE89230_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE89230_combined.write_csv(
    "../../Data/metadata/metadata_proseq/GSE89230_proseq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE89230_combined)

In [27]:
defined_pairs_gse89230 = [
    ("SRR4454569", "SRR4454567"), # PRO-seq K562 HS30, replicate 1 vs PRO-seq K562 NHS, replicate 1
    ("SRR4454570", "SRR4454568")  # PRO-seq K562 HS30, replicate 2 vs PRO-seq K562 NHS, replicate 2
]

paired_metadata_df_gse89230 = create_paired_metadata(
    combined_df=GSE89230_combined, 
    defined_sample_pairs=defined_pairs_gse89230,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse89230 is not None:
    paired_metadata_df_gse89230.write_csv(
        "../../Data/metadata/metadata_proseq/GSE89230_proseq_paired_metadata.tsv",
        separator='\t'
    )

display(paired_metadata_df_gse89230)

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,treatment,library_strategy,genome_build,sample_accession,rna_library,adapter,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_file_role,sra_aspera,fastq_galaxy,ncbi_reporting_standard,sample_description_right,sra_galaxy,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,sra_md5,fastq_ftp,base_count,library_strategy_right,library_source,sra_ftp,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,tag,first_created,scientific_name,tax_id,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR4454569""","""SRR4454567""","""pro-seq_k562_hs30_replicate_1 vs pro-seq_k562_nhs_replicate_1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR4454569 vs SRR4454567""","""SRX2271123 vs SRX2271121""","""PRO-seq K562 HS30, replicate 1 vs PRO-seq K562 NHS, replicate 1""","""GSM2361444 vs GSM2361442""","""Public on May 21 2017""","""Oct 27 2016""","""May 15 2019""","""SRA""","""1""","""K562 cells""","""Homo sapiens""","""9606""","""total RNA""","""Nascent RNA, Normalized_PRO-seq_K562_combined_replicates_HS30_plusStrand.bigWig, Normalized_PRO-seq_K562_combined_replicates_HS30_minusStrand.bigWig vs Nascent RNA, Normalized_PRO-seq_K562_combined_re…","""GPL11154""","""Anniina,,Vihervaara""","""mavih@kth.se""","""+358 50 5463694""","""Vihervaara""","""Gene Technology""","""KTH Royal Institute of Technology""","""Tomtebodavägen 23A""","""Stockholm""","""17165""","""Sweden""","""Illumina HiSeq 2000""","""other""","""transcriptomic""","""OTHER""","""GSM2361444 vs GSM2361442""","""NONE""","""GSE89230""","""0""","""SRX2271123 vs SRX2271121""","""30min at 42°C (HS30) vs Non-treated (NHS)""","""PRO-seq""","""hg19""","""SAMN05946060 vs SAMN05946058""","""Illumina TruSeq Small RNA Library Preparation Kits""","""Illumina TruSeq Small RNA adapters""","""SRR4454569 vs SRR4454567""","""Illumina HiSeq 2000 sequencing: GSM2361444: PRO-seq K562 HS30, replicate 1 Homo sapiens OTHER vs Illumina HiSeq 2000 sequencing: GSM2361442: PRO-seq K562 NHS, replicate 1 Homo sapiens OTHER""","""Homo sapiens""","""SRA487200""","""other""","""2017-05-23""","""7845288234 vs 4612741522""","""8852019321 vs 5430129171""","""ILLUMINA""","""ARCHIVAL_FILE""","""fasp.sra.ebi.ac.uk:/vol1/srr/SRR445/009/SRR4454569 vs fasp.sra.ebi.ac.uk:/vol1/srr/SRR445/007/SRR4454567""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR445/009/SRR4454569/SRR4454569.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR445/007/SRR4454567/SRR4454567.fastq.gz""","""Generic""","""PRO-seq K562 HS30, replicate 1 vs PRO-seq K562 NHS, replicate 1""","""ftp.sra.ebi.ac.uk/vol1/srr/SRR445/009/SRR4454569 vs ftp.sra.ebi.ac.uk/vol1/srr/SRR445/007/SRR4454567""","""469e919a92c4e603b18bf9846c

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 9. GSE112379 (SINGLE) PRO-seq

- ✅ DOWNLOAD (yone-SSD)
- ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD)
- ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD)

&nbsp;

- [Nascent-protein ubiquitination is required for heat shock–induced gene downregulation in human cells](https://doi.org/10.1038/s41594-018-0182-x)
- 熱ストレス下のデータが入手できるため、収集する
- 以下のような記載が論文にあった
- `Heat shock treatment: Unless otherwise stated, heat shock treatment was done by moving tissue-culture plates with growing cells from an incubator at normal growth temperature (37 °C) into an incubator at heat-shock temperatures (43 °C) for 1.5 h. Heat-shock treatment of K562 cells was done for 0.5 h. Heat-shock treatment of HeLa cells was done at 42 °C for the indicated times for activation and attenuation phase experiments29. To explore whether different heat-shock paradigms could influence the results obtained, we also applied acute heat shock, as described51. Briefly, two plates were grown up to 80% confluence. Medium from one of the plates was removed, preheated to 43 °C and added to the other plate from which the medium was withdrawn. The plate was then placed at 43 °C for 1.5 h.`
- HEK293 cellなので、43℃, 1.5hだと考えられる

&nbsp;

### Adapter

- [GSM3068957](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3068957) の `Extraction protocol` には以下のような記載がある
- `A 5’ RNA adaptor (VRA5) was ligated at the 5’ end of RNA using the ligation mix described previously.`
- VRA5 adapter sequence は、[Base-Pair Resolution Genome-Wide Mapping Of Active RNA polymerases using Precision Nuclear Run-On (PRO-seq)](https://doi.org/10.1038/nprot.2016.086) を見ると `CCUUGGCACCCGAGAAUUCCA` と書いてある
- これの逆相補鎖が `TGGAATTCTCGGGTGCCAAGG` になるので、Truseq Small RNA adapter に合致
- おそらく以下のような理解

```text
RNA:
5′-[VRA5]-[nascent RNA]-[VRA3]-3′
                         ← sequencing starts here

FASTQ read:
5′-[reverse complement of nascent RNA]-[reverse complement of VRA5]-3′
```


In [28]:
GSE112379_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE112379/GSE112379_PEP/GSE112379_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("organism") == "Homo sapiens") &
    (pl.col("protocol") != "ChIP")
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    (pl.lit("Custom PRO-seq library preparation based on Mahat et al. (2016)").alias("rna_library")),
    (pl.lit("https://doi.org/10.1038/s41594-018-0182-x").alias("reference_doi")),
    (pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")),
    (pl.lit("43 C").alias("stress treatment temperature")),
    (pl.lit("90 min.").alias("stress treatment time")),
    (pl.lit("37 C (inferred)").alias("control treatment temperature"))
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE112379 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE112379.metadata.tsv",
    separator='\t'
)

GSE112379_iseq_filtered = GSE112379[[s.name for s in GSE112379 if not (s.null_count() == GSE112379.height)]]


# display(GSE112379_PEP)

In [29]:
GSE112379_combined = GSE112379_PEP.join(
    GSE112379_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE112379_combined.write_csv(
    "../../Data/metadata/metadata_proseq/GSE112379_proseq_modified_metadata.tsv",
    separator='\t'
)

# pl.Config.set_tbl_rows(30)
# display(GSE112379_combined)

In [30]:
# Manual defined pairs
defined_pairs_gse112379 = [
    ("SRR6904129", "SRR6904125"), # HEK_HS_Proseq_Rep1 vs HEK_NHS_Proseq_Rep1
    ("SRR6904130", "SRR6904126"), # HEK_HS_Proseq_Rep2 vs HEK_NHS_Proseq_Rep2
    ("SRR6904131", "SRR6904127"), # HEK_HS_Proseq_Rep3 vs HEK_NHS_Proseq_Rep3
    ("SRR6904132", "SRR6904128") # HEK_HS_Proseq_Rep4 vs HEK_NHS_Proseq_Rep4
]

paired_metadata_df_gse112379 = create_paired_metadata(
    combined_df=GSE112379_combined,
    defined_sample_pairs=defined_pairs_gse112379,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse112379 is not None:
    paired_metadata_df_gse112379.write_csv(
        "../../Data/metadata/metadata_proseq/GSE112379_proseq_paired_metadata.tsv",
        separator='\t'  
    )
    display(paired_metadata_df_gse112379)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,gse_id,sample_data_row_count,experiment_accession,treatment,condition,library_strategy,genome_build,sample_accession,rna_library,reference_doi,adapter,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_file_role,sra_aspera,fastq_galaxy,ncbi_reporting_standard,sample_description,sra_galaxy,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,sra_md5,fastq_ftp,base_count,library_strategy_right,library_source,sra_ftp,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,tag,first_created,scientific_name,tax_id,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR6904129""","""SRR6904125""","""hek_hs_proseq_rep1 vs hek_nhs_proseq_rep1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR6904129 vs SRR6904125""","""SRX3852861 vs SRX3852857""","""HEK_HS_Proseq_Rep1 vs HEK_NHS_Proseq_Rep1""","""GSM3068954 vs GSM3068950""","""Public on Feb 07 2019""","""Mar 27 2018""","""Feb 07 2019""","""SRA""","""1""","""Flp-In T-REx 293 Cell Line""","""Homo sapiens""","""9606""","""Flp-In T-REx 293 were cultured in DMEM High Glucose supplemented with 10% FBS. K562 cells were cultured in IMDM supplemented with 10% FBS.""","""total RNA""","""10X106 HEK293 cells were used for each PROseq experiment. Cells were harvested and permeabilized. A 2-biotin label run on reaction was done to label nascently transcribed RNA. 2X nuclear run-on mix wa…","""ChIP-seq reads were aligned to the human genome build hg38 using Bowtie2. Duplicate and discordant reads were removed. Peak calling was done with MACS2 using ""--keep-dup all"", ""--nomodel"", ""--extsize""…","""GPL18573""","""Barbara,,Hummel""","""Stübeweg 51""","""Freiburg""","""79108""","""Germany""","""Illumina NextSeq 500""","""other""","""transcriptomic""","""OTHER""","""GSM3068954 vs GSM3068950""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3068nnn/GSM3068954/suppl/GSM3068954_HEK_HS_Proseq_Rep1.rpkm.mn.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3068nnn/GSM3068950/suppl/GSM3068950_HEK_NHS_Pro…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3068nnn/GSM3068954/suppl/GSM3068954_HEK_HS_Proseq_Rep1.rpkm.pl.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3068nnn/GSM3068950/suppl/GSM3068950_HEK_NHS_Pro…","""GSE112379""","""0""","""SRX3852861 vs SRX3852857""","""none""","""Heat-Shock vs Non Heat-Shock""","""Pro-Seq""","""hg38""","""SAMN08801007 vs SAMN08801012""","""Custom PRO-seq library preparation based on Mahat et al. (2016)""","""https://doi.org/10.1038/s41594-018-0182-x""","""Illumina TruSeq Small RNA adapters""","""43 C""","""90 min.""","""37 C (inferred)""","""SRR6904129 vs SRR6904125""","""NextSeq 500 sequencing: GSM30689

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 10. GSE144786 PRO-seq (SINGLE)

- ✅ DOWNLOAD (yone-SSD)
- ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD)
- ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD)

&nbsp;

- validate date: 2026-02-22
- [NELF Regulates a Promoter-Proximal Step Distinct from RNA Pol II Pause-Release](https://doi.org/10.1016/j.molcel.2020.02.014)
- 以下のような記述が論文にあるため、収集
- `Heat Shock Induction: Heat shock of mammalian cells was performed using ∼70%–80% confluent DLD-1 cells by adding pre-heated (42°C) conditioned media collected from identically growing cells (Mahat et al., 2016a). The heat shock cells were incubated at 42°C for 1 hour. After washing with PBS, the heat shock and non-heat shock DLD-1 cells were processed for PRO-seq as described below.`
- 熱ストレスに関連しているデータのみを選抜
- 熱ストレスと`Auxin`を処理しているデータ、`PRO-seq-20-NELFC-AID-42C-Aux-rep1`などが含まれており、悩んだ結果、Auxin処理されていないが、`-37C-`の記述があるデータとペアにすることに

&nbsp;

### Adapter

- [GSE144786](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE144786) のページで、例えば [GSM4296332](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4296332) の Extraction protocol には以下のように書いてある
- `For PRO-seq, adapter sequences previously described (Mahat et al., 2016) were used.`
- つまり、 Illumina TruSeq small-RNA adaptors に該当する

In [31]:
GSE144786_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE144786/GSE144786_PEP/GSE144786_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("organism") == "Homo sapiens") &
    (pl.col("protocol") != "ChIP") &
    (pl.col("sample_title").str.contains(r"-(37|42)C-")) # heat-shock related data
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    (pl.lit("unknown").alias("rna_library")),
    (pl.lit("https://doi.org/10.1016/j.molcel.2020.02.014").alias("reference_doi")),
    (pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")),
    (pl.lit("42 C").alias("stress treatment temperature")),
    (pl.lit("60 min.").alias("stress treatment time")),
    (pl.lit("37 C").alias("control treatment temperature"))
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE144786 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE144786.metadata.tsv",
    separator='\t'
)

GSE144786_iseq_filtered = GSE144786[[s.name for s in GSE144786 if not (s.null_count() == GSE144786.height)]]

# display(GSE144786_PEP)

In [32]:
GSE144786_combined = GSE144786_PEP.join(
    GSE144786_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE144786_combined.write_csv(
    "../../Data/metadata/metadata_proseq/GSE144786_proseq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE144786_combined)

In [33]:
# Manual defined pairs
defined_pairs_gse144786 = [
    ("SRR11024624", "SRR11024623"), # PRO-seq-19-NELFC-AID-42C-rep1 vs PRO-seq-18-NELFC-AID-37C-rep1
    ("SRR11024625", "SRR11024623"), # PRO-seq-20-NELFC-AID-42C-Aux-rep1 vs PRO-seq-18-NELFC-AID-37C-rep1 (Auxin処理されていないデータとペアにする)
    ("SRR11024627", "SRR11024626"), # PRO-seq-19-NELFC-AID-42C-rep2 vs PRO-seq-18-NELFC-AID-37C-rep2
    ("SRR11024628", "SRR11024626")  # PRO-seq-20-NELFC-AID-42C-Aux-rep2 vs PRO-seq-18-NELFC-AID-37C-rep2 (Auxin処理されていないデータとペアにする)
]

paired_metadata_df_gse144786_proseq = create_paired_metadata(
    combined_df=GSE144786_combined,
    defined_sample_pairs=defined_pairs_gse144786,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse144786_proseq is not None:
    paired_metadata_df_gse144786_proseq.write_csv(
        "../../Data/metadata/metadata_proseq/GSE144786_proseq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse144786_proseq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_type,genotype,antibody,cell_line,genome_build,sample_accession,rna_library,reference_doi,adapter,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR11024624""","""SRR11024623""","""pro-seq-19-nelfc-aid-42c-rep1 vs pro-seq-18-nelfc-aid-37c-rep1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR11024624 vs SRR11024623""","""SRX7677771 vs SRX7677770""","""PRO-seq-19-NELFC-AID-42C-rep1 vs PRO-seq-18-NELFC-AID-37C-rep1""","""GSM4296332 vs GSM4296331""","""Public on Mar 09 2020""","""Feb 05 2020""","""Mar 09 2020""","""SRA""","""1""","""DLD-1""","""Homo sapiens""","""9606""","""nuclear RNA""","""PRO-seq""","""GPL18573""","""Ali,,Shilatifard""","""ash@northwestern.edu""","""Shilatifard Lab""","""Department of Biochemistry and Molecular Genetics""","""320 E Superior St""","""Chicago""","""IL""","""60611""","""USA""","""Illumina NextSeq 500""","""other""","""transcriptomic""","""OTHER""","""GSM4296332 vs GSM4296331""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4296nnn/GSM4296332/suppl/GSM4296332_PRO-seq-19-NELFC-AID-42C-rep1.minus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4296nnn/GSM4296331/suppl/GSM4296331_PR…","""GSE144786""","""0""","""SRX7677771 vs SRX7677770""","""Colorectal adenocarcinoma""","""NELF-C-AID OsTIR1""","""null""","""DLD-1""","""hg19, dm3, mm9""","""SAMN14001984 vs SAMN14001972""","""unknown""","""https://doi.org/10.1016/j.molcel.2020.02.014""","""Illumina TruSeq Small RNA adapters""","""42 C""","""60 min.""","""37 C""","""SRR11024624 vs SRR11024623""","""NextSeq 500 sequencing: GSM4296332: PRO-seq-19-NELFC-AID-42C-rep1 Homo sapiens OTHER vs NextSeq 500 sequencing: GSM4296331: PRO-seq-18-NELFC-AID-37C-rep1 Homo sapiens OTHER""","""NELF regulates a promoter-proximal step distinct from RNA Pol II pause-release""","""SRA1037838""","""other""","""2020-03-23""","""2962047063 vs 6211627211""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR110/024/SRR11024624/SRR11024624.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR110/023/SRR11024623/SRR11024623.fastq.gz""","""Generic""","""PRO-seq-19-NELFC-AID-42C-rep1 vs PRO-seq-18-NELFC-AID-37C-rep1""","""5df7176074823743c942d10ca67e3a23 vs b63c0df75bf4470fdf00a222a7ce6eae""","""SRP247346""","""71153848 vs 147384362"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 11. GSE209685 PRO-seq (PAIRED)

- ✅ DOWNLOAD (yone-SSD)
- ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD)
- ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD)

&nbsp;

- validate date: 2026-02-23
- [Transcriptional responses of cancer cells to heat shock-inducing stimuli involve amplification of robust HSF1 binding](https://doi.org/10.1038/s41467-023-43157-7)
- 論文に以下のような記載があった
- `A 60-min incubation at 42 °C induced a control HSP gene seen as spreading of the Pol II and nascent RNA signal into the gene body (Fig. 1a).`
- こちらのデータセットにはヒ素 (As)処理のものも含まれているため、そのデータセットをフィルタリングして収集する

&nbsp;

### Adapter

- [GSE209685](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE209685) のページ、例えば[GSM6383642](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM6383642) の `Extraction protocol` には以下のように書いてある
- `Precision Run-On sequencing (PRO-seq) was performed as previously described {Mahat et al., 2016}. Briefly, nuclei were extracted from ~2x107 cells and run on reactions were carried on at 37°C for 3 minutes using 3ml of each 11-biotin-labelled ribonucleotide stocks (Perkin Elmer). Libraries were prepared as described in Mahat et al., 2016 using custom Illumina small RNA compatible linkers, alkaline degradation, enzymatic treatments and streptavidin selection of nascent RNA on magnetic beads according to the cited protocol.`
- つまり、Illumina Truseq small RNA adapters で大丈夫だと思われる

In [34]:
GSE209685_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE209685/GSE209685_PEP/GSE209685_PEP_raw.csv",
    separator=','
).filter(
    (~pl.col("sample_title").str.contains(r"As")) # ヒ素 (As)処理のデータをフィルタリング
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    (pl.lit("unknown").alias("rna_library")),
    (pl.lit("https://doi.org/10.1038/s41467-023-43157-7").alias("reference_doi")),
    (pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")),
    (pl.lit("42 C").alias("stress treatment temperature")),
    (pl.lit("60 min.").alias("stress treatment time")),
    (pl.lit("37 C (inferred)").alias("control treatment temperature"))
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE209685 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE209685.metadata.tsv",
    separator='\t'
)

GSE209685_iseq_filtered = GSE209685[[s.name for s in GSE209685 if not (s.null_count() == GSE209685.height)]]

# display(GSE209685_PEP)

In [35]:
GSE209685_combined = GSE209685_PEP.join(
    GSE209685_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
).sort(
    [
        "gsm_id"
    ],
    descending=[
        False
    ]
)

GSE209685_combined.write_csv(
    "../../Data/metadata/metadata_proseq/GSE209685_proseq_modified_metadata.tsv",
    separator='\t'
)

# pl.Config.set_tbl_rows(32)
# display(GSE209685_combined)

In [36]:
# Manual defined pairs
defined_pairs_gse209685 = [
    ("SRR20631364", "SRR20631365"), # K562_pro-seq_hs vs k562_pro-seq_nhs
    ("SRR20631361", "SRR20631362"), # mcf7_pro-seq_hs_rep1 vs mcf7_pro-seq_nhs_rep1
    ("SRR20631358", "SRR20631359")  # mcf7_pro-seq_hs_rep2 vs mcf7_pro-seq_nhs_rep2
]

paired_metadata_df_gse209685 = create_paired_metadata(
    combined_df=GSE209685_combined,
    defined_sample_pairs=defined_pairs_gse209685,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse209685 is not None:
    paired_metadata_df_gse209685.write_csv(
        "../../Data/metadata/metadata_proseq/GSE209685_proseq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse209685)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,assembly,library_strategy,sample_accession,rna_library,reference_doi,adapter,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_file_role,sra_aspera,fastq_galaxy,country,ncbi_reporting_standard,sample_description_right,sra_galaxy,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,sra_md5,cell_type_right,fastq_ftp,base_count,library_strategy_right,library_source,sra_ftp,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR20631364""","""SRR20631365""","""k562_pro-seq_hs vs k562_pro-seq_nhs""","""null""","""null""","""null""","""null""","""null""","""null""","""K562 PRO-seq HS vs K562 PRO-seq NHS""","""GSM6383642 vs GSM6383641""","""Public on Oct 30 2023""","""Jul 25 2022""","""Oct 30 2023""","""SRA""","""1""","""K562""","""Homo sapiens""","""9606""","""total RNA""","""nascent RNA""","""GPL24676""","""Sergei,,Nechaev""","""sergei.nechaev@und.edu""","""Nechaev Lab""","""Biomedical Sciences""","""University of North Dakota""","""Grand Forks""","""North Dakota""","""58202""","""USA""","""Illumina NovaSeq 6000""","""other""","""transcriptomic""","""OTHER""","""GSM6383642 vs GSM6383641""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383642/suppl/GSM6383642_K562_HS_PROseq_for.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383641/suppl/GSM6383641_K562_NHS_PROseq_for…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383642/suppl/GSM6383642_K562_HS_PROseq_rev.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383641/suppl/GSM6383641_K562_NHS_PROseq_rev…","""GSE209685, GSE209687""","""0""","""SRX16654921 vs SRX16654920""","""K562""","""bone marrow, myelogenous leukemia""","""Heat shock vs Untreated""","""hg19""","""PRO-seq""","""SAMN29939165 vs SAMN29939166""","""unknown""","""https://doi.org/10.1038/s41467-023-43157-7""","""Illumina TruSeq Small RNA adapters""","""42 C""","""60 min.""","""37 C (inferred)""","""SRR20631364 vs SRR20631365""","""Illumina NovaSeq 6000 sequencing: GSM6383642: K562 PRO-seq HS Homo sapiens OTHER vs Illumina NovaSeq 6000 sequencing: GSM6383641: K562 PRO-seq NHS Homo sapiens OTHER""","""Transcriptional responses of cancer cells to heat shock-inducing stimuli involve amplification of robust HSF1 binding""","""SRA1462089""","""other""","""2023-11-02""","""13754988616 vs 9321066699""","""3923054022;6067302367 vs 2780826311;4188345667"""

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 12. GSE200269 PRO-IP-seq (SINGLE)

<!-- - ✅ DOWNLOAD (yone-SSD) -->
<!-- - ✅ Preprocesses PRO-seq data using [proseq2.0.bsh](https://github.com/Danko-Lab/proseq2.0) (yone-SSD) -->
<!-- - ✅ Caluculate average scores using [bigWigAverageOverBed](https://open.bioqueue.org/home/knowledge/showKnowledge/sig/ucsc-bigwigaverageoverbed) as fanta.bio CRE level (yone-SSD) -->

&nbsp;

- [PRO-IP-seq tracks molecular modifications of engaged Pol II complexes at nucleotide resolution](https://doi.org/10.1038/s41467-023-42715-3)
- [./12_8_metadata_other.ipynb](./12_8_metadata_other.ipynb) から処理を移動
- PRO-IP-seq という C-terminal Domain にリン酸化修飾を受けている RNA Polymerase II が持つ nascent RNA を検出している手法
- GEOのページの Treatment protocol に以下のような記載がある
- `The cells were maintained at 37°C (NHS) or heat treated for 30 min at 42°C (HS30). The heat shock was conducted in pre-conditioned pre-warmed media. The control cells (NHS) were treated as the heat shocked (HS30) cells but using preconditioned media pre-warmed to 37°C.`
- 

In [37]:
GSE200269_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE200269/GSE200269_PEP/GSE200269_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_organism_ch1") == "Homo sapiens") &
    (pl.col("library_preparation") == "PRO-IP-seq")
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    # (pl.lit("unknown").alias("rna_library")),
    (pl.lit("https://doi.org/10.1038/s41467-023-43157-7").alias("reference_doi")),
    # (pl.lit("Illumina TruSeq Small RNA adapters").alias("adapter")),
    (pl.lit("42 C").alias("stress treatment temperature")),
    (pl.lit("30 min.").alias("stress treatment time")),
    (pl.lit("37 C").alias("control treatment temperature"))
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample",
    "organism",
    "read_type",
    "data_source"
)


# metadata from iseq
GSE200269 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE200269.metadata.tsv",
    separator='\t'
)

GSE200269_iseq_filtered = GSE200269[[s.name for s in GSE200269 if not (s.null_count() == GSE200269.height)]]


# display(GSE200269_PEP)

In [38]:
GSE200269_combined = GSE200269_PEP.join(
    GSE200269_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
).with_columns(
    (pl.col("run_accession").alias("srr")), # run_accession の値を srr カラムに上書き
    (pl.col("experiment_accession").alias("srx")) # sample_title の値を sample_name カラムに上書き
).sort(
    [
        "gsm_id"
    ],
    descending=[
        False
    ]
)

GSE200269_combined.write_csv(
    "../../Data/metadata/metadata_proseq/GSE200269_pro-ip-seq_modified_metadata.tsv",
    separator='\t'
)

# pl.Config.set_tbl_rows(32)
# display(GSE200269_combined)

In [39]:
# Manual defined pairs
defined_pairs_gse200269 = [
    # i1 (Downloaded from SRA)
    ("SRR18653122", "SRR18653125"), # i1_hs30_ctd_rep1 vs. i1_nhs_ctd_rep1 (unphosphorylated CTD (unPhCTD))
    ("SRR18653111", "SRR18653121"), # i1_hs30_igg_rep1 vs. i1_nhs_igg_rep1 (unspecific IgG)
    ("SRR18653112", "SRR18653118"), # i1_hs30_input_rep1 vs. i1_nhs_input_rep1 (no antibody = PRO-seq)
    ("SRR18653117", "SRR18653124"), # i1_hs30_s2_rep1 vs. i1_nhs_s2_rep1 (serine-2-phosphorylated CTD (ser2PhCTD))
    ("SRR18653123", "SRR18653120"), # i1_hs30_s5_rep1 vs. i1_nhs_s5_rep1 (serine-5-phosphorylated CTD (ser5PhCTD))
    ("SRR18653116", "SRR18653119"), # i1_hs30_s7_rep1 vs. i1_nhs_s7_rep1 (serine-7-phosphorylated CTD (ser7PhCTD))

    # i2
    ("SRR18653115", "SRR18653110"), # i2_hs30_ctd_rep2 vs. i2_nhs_ctd_rep2 (unphosphorylated CTD (unPhCTD))
    ("SRR18653101", "SRR18653114"), # i2_hs30_igg_rep2 vs. i2_nhs_igg_rep2 (unspecific IgG)
    ("SRR18653100", "SRR18653104"), # i2_hs30_input_rep2 vs. i2_nhs_input_rep2 (no antibody = PRO-seq)
    ("SRR18653103", "SRR18653113"), # i2_hs30_s2_rep2 vs. i2_nhs_s2_rep2 (serine-2-phosphorylated CTD (ser2PhCTD))
    ("SRR18653102", "SRR18653109"), # i2_hs30_s5_rep2 vs. i2_nhs_s5_rep2 (serine-5-phosphorylated CTD (ser5PhCTD))
    # なぜか i2_hs30_s7_rep2 に該当するような accession が メタデータには登録されていない...

    # i3
    ("SRR18653094", "SRR18653099"), # i3_hs30_ctd_rep3 vs. i3_nhs_ctd_rep3 (unphosphorylated CTD (unPhCTD))
    # なぜか i3_hs30_igg_rep3 に該当するような accession が メタデータには登録されていない...
    ("SRR18653107", "SRR18653095"), # i3_hs30_input_rep3 vs. i3_nhs_input_rep3 (no antibody = PRO-seq)
    ("SRR18653108", "SRR18653098"), # i3_hs30_s2_rep3 vs. i3_nhs_s2_rep3 (serine-2-phosphorylated CTD (ser2PhCTD))
    ("SRR18653106", "SRR18653097"), # i3_hs30_s5_rep3 vs. i3_nhs_s5_rep3 (serine-5-phosphorylated CTD (ser5PhCTD))
    ("SRR18653105", "SRR18653096")  # i3_hs30_s7_rep3 vs. i3_nhs_s7_rep3 (serine-7-phosphorylated CTD (ser7PhCTD))
]

paired_metadata_df_gse200269 = create_paired_metadata(
    combined_df=GSE200269_combined,
    defined_sample_pairs=defined_pairs_gse200269,
    sample_id_column_name="run_accession"
)

if paired_metadata_df_gse200269 is not None:
    paired_metadata_df_gse200269.write_csv(
        "../../Data/metadata/metadata_proseq/GSE200269_pro-ip-seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse200269)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,antibody,antibody_epitope,library_preparation,assembly,sample_accession,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,study_accession,secondary_study_accession,secondary_sample_accession,submission_accession,tax_id,scientific_name,instrument_platform,instrument_model,library_name,library_layout,library_strategy,library_source,library_selection,read_count,base_count,first_public,last_updated,experiment_title,study_title,study_alias,experiment_alias,run_alias,fastq_bytes,fastq_md5,fastq_ftp,fastq_aspera,fastq_galaxy,sample_alias,sample_title_right,first_created
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR18653122""","""SRR18653125""","""i1_hs30_ctd_rep1 vs i1_nhs_ctd_rep1""","""Heat treated at 42degreeC for 30 minutes (HS30) vs No heat shock, maintained at 37degreeC (NHS)""","""SRR18653122 vs SRR18653125""","""SRX14756151 vs SRX14756145""","""i1_HS30_CTD_rep1 vs i1_NHS_CTD_rep1""","""GSM6025799 vs GSM6025793""","""Public on Feb 24 2023""","""Apr 06 2022""","""Feb 24 2023""","""SRA""","""1""","""K562""","""Homo sapiens""","""9606""","""total RNA""","""Illumina NextSeq 500 and Illumina NovaSeq6000""","""nascent RNA, iPRO_HS30_CTD_BRs_normalized_pl.bigWig, iPRO_HS30_CTD_BRs_normalized_mn.bigWig vs nascent RNA, iPRO_NHS_CTD_BRs_normalized_pl.bigWig, iPRO_NHS_CTD_BRs_normalized_mn.bigWig""","""GPL18573""","""Anniina,,Vihervaara""","""mavih@kth.se""","""+358 50 5463694""","""Vihervaara""","""Gene Technology""","""KTH Royal Institute of Technology""","""Tomtebodavägen 23A""","""Stockholm""","""17165""","""Sweden""","""Illumina NextSeq 500""","""other""","""transcriptomic""","""RIP-Seq""","""GSM6025799 vs GSM6025793""","""NONE""","""GSE200269""","""0""","""SRX14756151 vs SRX14756145""","""AbCam YWG16""","""unphosphorylated CTD (unPhCTD)""","""PRO-IP-seq""","""hg19""","""SAMN27365889 vs SAMN27365895""","""https://doi.org/10.1038/s41467-023-43157-7""","""42 C""","""30 min.""","""37 C""","""PRJNA823909""","""SRP367808""","""SRS12517005 vs SRS12516999""","""SRA1398645""","""9606""","""Homo sapiens""","""ILLUMINA""","""NextSeq 500""","""GSM6025799 vs GSM6025793""","""SINGLE""","""RIP-Seq""","""TRANSCRIPTOMIC""","""other""","""43462412 vs 17682128""","""3571008881 vs 1454223425""","""2023-02-27""","""2023-02-28 vs 2023-03-14""","""NextSeq 500 sequencing: GSM6025799: i1_HS30_CTD_rep1 Homo sapiens RIP-Seq vs NextSeq 500 sequencing: GSM6025793: i1_NHS_CTD_rep1 Homo sapiens RIP-Seq""","""PRO-IP-seq: Pol II CTD code at nt-resolution in human K562 cells""","""GSE200269""","""GSM6025799_r1 vs GSM6025793_r1""","""GSM6025799_r1 vs GSM6025793_r1""","""1580627852 vs 392995677""","""f377febd86b4d327de6a411b4d03a133 vs cdd794619cecbfa51ddca8afa484dc08""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR186/022/SRR18653122/SRR18653122.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR186/025/SRR18653125/SRR18653125.fastq.gz""","""fasp.sra.ebi.ac.uk:/vol1/fastq/SRR186/022/SRR18653122/SRR18653122.fastq.gz vs fasp.sra.ebi.ac

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## concat dataframe (including GRO-seq)

In [40]:
dfs_proseq_all: list[pl.DataFrame | None] = [
    paired_metadata_gse228346_proseq,
    paired_metadata_df_gse173537_pro_seq,
    paired_metadata_df_gse211651_pro_seq,
    paired_metadata_df_gse154746_pro_seq,
    paired_metadata_df_gse202187,
    paired_metadata_df_gse127844_pro_seq,
    paired_metadata_df_gse66448, # GRO-seq
    paired_metadata_df_gse89230,
    paired_metadata_df_gse112379,
    paired_metadata_df_gse144786_proseq,
    paired_metadata_df_gse209685,
    # paired_metadata_df_gse200269 # PRO-IP-seq
]


dfs_proseq:  list[pl.DataFrame] = [df for df in dfs_proseq_all if df is not None]

In [41]:
proseq_concat = pl.concat(dfs_proseq, how="diagonal_relaxed")

proseq_concat.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/proseq_metadata_all_{execute_date}.tsv",
    separator='\t'
)